In [1]:
from meshparty import *
import orjson
import numpy as np
import pandas as pd
from caveclient import CAVEclient

### Download data

In [2]:
# client = CAVEclient('minnie65_phase3_v1')
# tbl_qry = client.materialize.tables

# cell_df = tbl_qry.allen_column_mtypes_v2(cell_type="PTC").query(
#     desired_resolution=[1,1,1]
# )
# root_id = cell_df.iloc[10].pt_root_id
# with open("root_id.json", "w") as f:
#     f.write(
#         str(root_id)
#     )
# pre_syn_df = tbl_qry.synapses_pni_2(pre_pt_root_id=root_id).query(desired_resolution=[1,1,1], split_positions=True)
# post_syn_df = tbl_qry.synapses_pni_2(post_pt_root_id=root_id).query(desired_resolution=[1,1,1], split_positions=True)

# pre_syn_df.to_feather('pre.feather')
# post_syn_df.to_feather('post.feather')

# skel = client.skeleton.get_skeleton(root_id)
# with open('skel.json', 'bw') as f:
#     f.write(
#         orjson.dumps(skel, option=orjson.OPT_SERIALIZE_NUMPY)
#     )
# l2ids = skel['lvl2_ids']
# l2_df = client.l2cache.get_l2data_table(l2ids)
# l2_df.to_feather('l2properties.feather')
# l2_graph = client.chunkedgraph.level2_chunk_graph(root_id)
# with open("l2graph.json", "wb") as f:
#     f.write(
#         orjson.dumps(l2_graph, option=orjson.OPT_SERIALIZE_NUMPY)
#     )

### Load data locally

In [3]:
with open('root_id.json', 'r') as f:
    root_id = int(f.read())

pre_syn_df = pd.read_feather(
    'pre.feather'
)
post_syn_df = pd.read_feather(
    'post.feather',
)

with open('skel.json') as f:
    skel = orjson.loads(f.read())

with open('l2graph.json') as f:
    l2_graph = orjson.loads(f.read())

l2_df = pd.read_feather('l2properties.feather')
l2_df.reset_index(inplace=True)

In [4]:
import morphsync as sync

In [5]:
import fastremap
spatial_columns = ['ctr_pt_position_x', 'ctr_pt_position_y', 'ctr_pt_position_z']

properties = [x for x in l2_df.reset_index().columns.values if x not in spatial_columns]
l2_map = {v:k for k,v in l2_df['l2_id'].to_dict().items()}
edges = fastremap.remap(
    l2_graph,
    l2_map,
)
l2_spatial_columns = ['rep_coord_nm_x', 'rep_coord_nm_y', 'rep_coord_nm_z',]

In [6]:
l2_df_reidx = l2_df.set_index('l2_id')

In [7]:
cell = sync.MorphSync()
cell.add_graph(
    graph=(l2_df_reidx, edges),
    name='graph',
    spatial_columns=l2_spatial_columns,
)
cell.add_graph(
    graph=(np.array(skel['vertices']), np.array(skel['edges'])),
    name='skeleton',
)
cell.add_link(
    source='graph',
    target='skeleton',
    mapping=np.array(skel['mesh_to_skel_map']),
)
new_cell = cell.apply_mask(
    'skeleton',
    mask=np.array(skel['compartment'])==2,
)
new_cell._layers

{'skeleton': Graph(nodes=(14153, 3), edges=(14152, 2)),
 'graph': Graph(nodes=(21030, 27), edges=(0, 2))}

In [8]:
pc = PointCloudSync(
    'pre_syn',
    pre_syn_df,
    spatial_columns=spatial_columns,
)

In [9]:
GraphSync(
    name='l2_graph',
    vertices=l2_df,
    spatial_columns=l2_spatial_columns,
    edges=edges,
    vertex_index='l2_id',
)

GraphSync(name=l2_graph, vertices=25889, edges=30352)

In [10]:
nrn = MeshWorkSync(
    name=root_id,
)

nrn.add_graph(
    vertices=l2_df,
    spatial_columns=l2_spatial_columns,
    edges=edges,
    vertex_index="l2_id",
)

nrn.add_skeleton(
    vertices=np.array(skel['vertices']),
    edges=np.array(skel['edges']),
    linkage=Link(mapping=skel['mesh_to_skel_map'], source='graph', map_value_is_index=False)
)

MeshWork(name=864691135101289504, graph+skel, annotations=[])

In [11]:
nrn._morphsync.apply_mask(
    'skeleton',
    mask=np.array(skel['compartment'])==3,
).layers

,layer,layer_type
name,,
skeleton,"Graph(nodes=(2566, 3), edges=(2564, 2))",Graph
graph,"Graph(nodes=(4742, 27), edges=(5801, 2))",Graph


In [12]:
from caveclient import CAVEclient
client = CAVEclient('minnie65_phase3_v1')
ts = client.chunkedgraph.get_root_timestamps(root_id, latest=True)[0]
l2_ids = client.chunkedgraph.get_roots(pre_syn_df['pre_pt_supervoxel_id'], stop_layer=2, timestamp=ts)
pre_syn_df['pre_pt_l2_id'] = l2_ids

In [13]:
nrn.add_point_annotations(
    'pre_syn',
    vertices=pre_syn_df,
    spatial_columns=['ctr_pt_position_x', 'ctr_pt_position_y', 'ctr_pt_position_z'],
    vertex_index='id',
    linkage=Link(mapping='pre_pt_l2_id', target='graph')
)

MeshWork(name=864691135101289504, graph+skel, annotations=['pre_syn'])

In [16]:
nrn._morphsync.apply_mask(
    'skeleton',
    mask=np.array(skel['compartment'])==2,
).layers

,layer,layer_type
name,,
skeleton,"Graph(nodes=(14153, 3), edges=(14152, 2))",Graph
graph,"Graph(nodes=(21030, 27), edges=(24296, 2))",Graph
pre_syn,"Points(points=(6477, 3))",Points


In [ ]:
nrn._morphsync.add_link(
    source='graph',
    target='skeleton',
    mapping=np.array(skel['mesh_to_skel_map']),
)


In [ ]:
nrn.skeleton.add_label(
    skel['compartment'],
    name='compartment'
)

In [ ]:
new_morphsync = nrn.skeleton._morphsync.apply_mask(
    layer_name='skeleton',
    mask=(nrn.skeleton.labels['compartment']==2).values
)

In [ ]:
nrn.graph

In [ ]:
new_morphsync.graph

In [ ]:
nrn.skeleton.apply_mask(
    mask=(nrn.skeleton.labels['compartment']==2).values
)

In [ ]:
nrn._morphsync.layers

In [ ]:
nrn._morphsync.add_link(source=)

In [ ]:
nrn.

In [ ]:
client = CAVEclient('minnie65_phase3_v1')

In [ ]:
ts = client.chunkedgraph.get_root_timestamps(root_id, latest=True)

In [ ]:
l2ids = client.chunkedgraph.get_roots(pc.nodes['pre_pt_supervoxel_id'], stop_layer=2, timestamp=ts[0])

In [ ]:
pc.add_label(l2ids, name='l2id_pre')

In [ ]:
syn_index_map = [l2_map[x] for x in l2ids if x in l2_map]

In [ ]:
nrn.add_point_annotations(
    'pre_syn',
    vertices=pc,
    linkage={'graph': np.array(syn_index_map)}
)

In [ ]:
nrn._morphsync

In [ ]:
nrn.annotations.names()

In [ ]:
nrn.

In [ ]:
nrn.add_skeleton(
    vertices=skel['vertices'],
    edges=skel['edges'],
    labels={'radius': skel['radius']},
)

In [ ]:
nrn._annotations.names()

In [ ]:
l2_df.reset_index(inplace=True)

In [ ]:
l2_df

In [ ]:
nrn.add_graph(
    vertices=
    spatial_columns=spatial_columns,
    edges=edges,
    properties=properties,
)

In [ ]:
l2_df['l2_id'].values

In [ ]:
%%timeit


In [ ]:
skel['mesh_to_skel_map']

In [ ]:
fastremap.renumber(
    l2_df.index.values,
)

In [ ]:
fastremap.remap(
    l2_graph,
)

In [ ]:
nrn.skeleton.labels

In [ ]:
pre_syn_df

In [ ]:
nrn.add_point_annotations(
    'pre_syn',
    vertices=pre_syn_df,
    spatial_columns=spatial_columns,
)

In [ ]:
nrn._morphsync.add_graph?

In [ ]:
pc.labels['size']

In [ ]:
mask = pc.get_label('size') > 4400

In [ ]:
pcm = pc._apply_mask(mask)

In [ ]:
pcm._links

In [ ]:
pc._morphsync.layers.loc[pc.name].layer

In [ ]:
layer = pc._morphsync.layers.loc[pc.name]

In [ ]:
layer = pc._morphsync.layers.loc[pc.name].layer

In [ ]:
pc._morphsync.

In [ ]:
pc._morphsync._layers.items()

In [ ]:
pc._morphsync._generate_new_morphology(pc.name, layer.vertices_index)

In [ ]:
pd.concat?

In [ ]:
pc._morphsync

In [ ]:
pc._morphsync.layers.loc['pre_syn'].layer

In [ ]:
knp.vstack(
    pre_syn_df[].values
)

In [ ]:
PointCloud(pre_syn_df)

In [ ]:
pre_syn_df.columns[~pre_syn_df.columns.isin(spatial_columns)]

In [ ]:
list(pd.DataFrame().columns)